In [ ]:
import json
import pandas as pd
import datasets

In [ ]:
HF_DATASET_NAME = "blind-spots-neurips/blind-spots-bench"

In [ ]:
df = pd.read_excel("../data/reasoning_course.xlsx", sheet_name="dataset-0319")
print("Original length:", len(df))

df.loc[df["QID"] == 109,"valid_flag"] = 0

# Filter invalid questions
print(f"Removing {len(df[df['valid_flag'] != 1])} invalid questions.")
df = df[df['valid_flag'] == 1].copy()
print(f"Remaining valid questions: {len(df)}")

In [ ]:
# Create a map from 'author' name to hash
import hashlib
unique_authors = df['author'].dropna().unique()
author_map = {author: hashlib.sha256(str(author).encode()).hexdigest()[:8] for author in unique_authors}

# Replace author names with their hashes in the dataframe
df['author'] = df['author'].map(author_map)

In [ ]:
import os
import shutil


for index, row in df.iterrows():
    if row["question_type"] in ["multi-to-image", "multi-to-text"]:
        if pd.isna(row['img_path']):
            print(f"Error: Missing image path for QID {row['QID']}")
            continue
        else:
            # Check the existence of the image file
            full_img_path = os.path.join("../data/images/", row['img_path'])
            if os.path.isfile(full_img_path):
                # Overwrite the complete image path in the dataframe
                df.at[index, 'img_path'] = full_img_path
            else:
                print(f"Error: Image file not found for QID {row['QID']} at path {row['img_path']}")
    else:
        if not pd.isna(row['img_path']):
            print(f"Warning: Question {row['QID']} of type '{row['question_type']}' should not have an image path, but found '{row['img_path']}'.")

In [ ]:
df.drop(columns=["missing_image", "sheet", "valid_reason", "valid_flag", "comments", "percentages", "occurances", "Types"], inplace=True)
df.rename(columns={"sub-task type": "sub-task_type", "Failure mode": "failure_mode"}, inplace=True)
df["Re_Eval"] = df["Re_Eval"].fillna(0.0).astype(bool)
df["QID"] = df.apply(lambda row: row["QID"] + 1000 if row["Re_Eval"] else row['QID'], axis=1).astype(int)

In [ ]:
# Reduce size of images to max 1024x1024 while maintaining aspect ratio
from PIL import Image

max_size = (1024, 1024)

for index, row in df.iterrows():
    if pd.notna(row['img_path']):
        try:
            with Image.open(row['img_path']) as img:
                # Check if resize is needed
                if img.size[0] > max_size[0] or img.size[1] > max_size[1]:
                    img.thumbnail(max_size)
                    img.save(row['img_path'])
        except Exception as e:
            print(f"Error processing image at '{row['img_path']}' (question {row['QID']}): {e}")

In [ ]:
# Merge question types multiple nomenclature into one
def qtype_merge(qtype):
	if qtype in ["text-to-multi", "multi-to-image", "text-to-image"]:
		return "text-to-image"
	return qtype

df['question_type'] = df['question_type'].apply(qtype_merge)

In [ ]:
def clean_link_field(link):
	values_to_remove = [
		"(Gemini 2.5 Flash)",
		"(ChatGPT 5)",
		"(Gemini 2.5 Pro)\n",
		" (Gemini Pro 2.5)\n",
		" (Gemini 2.5 Flash)\n",
	]
	for v in values_to_remove:
		if v in link:
			link = link.split(v)[0]

	return link.strip()


df['link'] = df['link'].apply(clean_link_field)

# Drop rows where both 'prompt' and 'image' are missing
initial_len = len(df)
df = df[~(df['prompt'].isna() & df['img_path'].isna())]
print(f"Removed {initial_len - len(df)} rows with both 'prompt' and 'image' missing")
df['prompt'] = df['prompt'].fillna('')

In [ ]:
# Create a Dataset from the DataFrame and push to huggingface hub
from datasets import Dataset, Image, DatasetDict

# Rename img_path to image for standard naming
if 'img_path' in df.columns:
    df = df.rename(columns={'img_path': 'image'})

def get_image_path(path):
    if isinstance(path, str) and os.path.exists(path):
        return path
    return None

df['image'] = df['image'].apply(get_image_path)

### Create re-eval dataset for latest changes re-evaluation

In [ ]:
# Keep in the delta dataset only the problem with "Re_Eval" set
delta_df = df[df["Re_Eval"] == True]

# Drop the column
delta_df.drop(columns=["Re_Eval"], inplace=True)
df.drop(columns=["Re_Eval"], inplace=True)


# Create main dataset
ds = Dataset.from_pandas(df, preserve_index=False)
# Cast the image column to Image feature
ds = ds.cast_column("image", Image())
df_dict = DatasetDict({"test": ds})

# Create delta dataset
delta_ds = Dataset.from_pandas(delta_df, preserve_index=False)
delta_ds = delta_ds.cast_column("image", Image())
delta_df_dict = DatasetDict({"test": delta_ds})


print(ds)
print(delta_ds)

### Show a diff before pushing

In [ ]:
old_ds = datasets.load_dataset(f"{HF_DATASET_NAME}", split="test")

In [ ]:
df = ds.to_pandas()
old_df = old_ds.to_pandas()

# Merge
merged = pd.merge(df, old_df, on="QID", how="inner", suffixes=('_new', '_old'), indicator=True)
print(f"New: {len(df)} - Old: {len(old_df)} - Merged: {len(merged)}")

# Check for differences in the merged dataset
for i, row in merged.iterrows():
	has_diff = False
	for col in df.columns:
		if col == "image":
			continue
		new_col = f"{col}_new"
		old_col = f"{col}_old"
		if new_col in merged.columns and old_col in merged.columns:
			if row[new_col] != row[old_col]:
				if not has_diff:
					print(f"Difference in QID {row['QID']}:")
					has_diff = True
				print(f"  Column: {col}\t {row[old_col]} -> {row[new_col]}")
	if has_diff:
		print()

### Check distribution before pushing

In [ ]:
print(df["question_type"].value_counts())
print(delta_df["question_type"].value_counts())

## Push to hub

In [ ]:
# Push to hub
df_dict.push_to_hub(f"{HF_DATASET_NAME}")

In [ ]:
delta_df_dict.push_to_hub(f"{HF_DATASET_NAME}-delta", private=True)